In [1]:
import pandas as pd

print("Loading game summary to identify 2018+ games...")
game_summary = pd.read_csv('game_summary.csv')

# Filter for seasons 2018 and later
modern_games = game_summary[game_summary['season'] >= 2018]['game_id'].unique()
print(f"Targeting {len(modern_games)} games from the 2018-2023 modern era.")

print("Processing play_by_play.csv in chunks...")
chunk_size = 100_000
filtered_chunks = []

for chunk in pd.read_csv('play_by_play.csv', chunksize=chunk_size, low_memory=False):
    filtered_chunk = chunk[chunk['game_id'].isin(modern_games)]
    filtered_chunks.append(filtered_chunk)

final_pbp = pd.concat(filtered_chunks, ignore_index=True)
final_pbp.to_csv('play_by_play_filtered.csv', index=False)
print("Success! Filtered file saved as 'play_by_play_filtered.csv'.")

Loading game summary to identify 2018+ games...
Targeting 5855 games from the 2018-2023 modern era.
Processing play_by_play.csv in chunks...
Success! Filtered file saved as 'play_by_play_filtered.csv'.


In [2]:
import gzip
import shutil

print("Compressing play_by_play_filtered.csv...")
with open('play_by_play_filtered.csv', 'rb') as f_in:
    with gzip.open('play_by_play_filtered.csv.gz', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
        
print("Success! Compressed file saved as 'play_by_play_filtered.csv.gz'.")

Compressing play_by_play_filtered.csv...
Success! Compressed file saved as 'play_by_play_filtered.csv.gz'.


In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the compressed filtered dataset directly
df = pd.read_csv("play_by_play_filtered.csv.gz")

print(f"Successfully loaded {len(df):,} play-by-play rows!")

Successfully loaded 2,494,547 play-by-play rows!


In [5]:
# Filter for actual shot events only (eventmsgtype 1 = made shot, 2 = missed shot)
shots_df = df[df['eventmsgtype'].isin([1, 2])].copy()

# Create play_desc by combining home and visitor descriptions
shots_df['play_desc'] = shots_df['homedescription'].fillna(shots_df['visitordescription'])
shots_df['clean_desc'] = shots_df['play_desc'].fillna('').str.lower()

# Identify shot outcomes
shots_df['is_made'] = np.where(shots_df['eventmsgtype'] == 1, 1, 0)
shots_df['is_3pt'] = np.where((shots_df['is_made'] == 1) & (shots_df['clean_desc'].str.contains('3pt', na=False)), 1, 0)
shots_df['points_earned'] = np.select(
    [shots_df['is_3pt'] == 1, shots_df['is_made'] == 1],
    [3, 2],
    default=0
)

# Compute macro totals on shots only
total_fga = len(shots_df)
total_fgm = shots_df['is_made'].sum()
total_3pm = shots_df['is_3pt'].sum()
total_points = shots_df['points_earned'].sum()

# Compute empirical efficiency rates
fg_pct = round(total_fgm / total_fga, 3)
efg_pct = round((total_fgm + (0.5 * total_3pm)) / total_fga, 3)
pps = round(total_points / total_fga, 3)

print("--- Data Validation Summary (Shots Only) ---")
print(f"Total Attempts (FGA): {total_fga:,}")
print(f"Total Made (FGM):     {total_fgm:,}")
print(f"Field Goal %:         {fg_pct}")
print(f"Effective FG % (eFG%): {efg_pct}")
print(f"Points Per Shot (PPS): {pps}")

--- Data Validation Summary (Shots Only) ---
Total Attempts (FGA): 941,899
Total Made (FGM):     436,750
Field Goal %:         0.464
Effective FG % (eFG%): 0.533
Points Per Shot (PPS): 1.066


In [6]:
# Filter for actual shot events only
shots_df = df[df['eventmsgtype'].isin([1, 2])].copy()

# Create play_desc and clean description
shots_df['play_desc'] = shots_df['homedescription'].fillna(shots_df['visitordescription'])
shots_df['clean_desc'] = shots_df['play_desc'].fillna('').str.lower()

# Map Spatial Zones
conditions = [
    shots_df['clean_desc'].str.contains('layup|dunk|tip', regex=True, na=False),
    shots_df['clean_desc'].str.contains('3pt', na=False) & shots_df['clean_desc'].str.contains('left side|left corner', regex=True, na=False),
    shots_df['clean_desc'].str.contains('3pt', na=False) & shots_df['clean_desc'].str.contains('right side|right corner', regex=True, na=False),
    shots_df['clean_desc'].str.contains('3pt', na=False),
    shots_df['clean_desc'].str.contains('floating|hook|bank', regex=True, na=False)
]
choices = [
    '1. Restricted Area (0-4ft)',
    '4. Left Corner 3',
    '5. Right Corner 3',
    '6. Above-the-Break 3',
    '2. Paint / Short Mid (5-14ft)'
]
shots_df['spatial_zone'] = np.select(conditions, choices, default='3. Long Mid-Range (15ft+)')

# Identify outcomes & points
shots_df['is_made'] = np.where(shots_df['eventmsgtype'] == 1, 1, 0)
shots_df['is_3pt'] = np.where((shots_df['is_made'] == 1) & (shots_df['clean_desc'].str.contains('3pt', na=False)), 1, 0)
shots_df['points_earned'] = np.select([shots_df['is_3pt'] == 1, shots_df['is_made'] == 1], [3, 2], default=0)

# Group by zone and aggregate metrics
zone_summary = shots_df.groupby('spatial_zone').agg(
    total_fga=('is_made', 'count'),
    total_fgm=('is_made', 'sum'),
    total_3pm=('is_3pt', 'sum'),
    total_points=('points_earned', 'sum')
).reset_index()

# Calculate eFG% and PPS per zone
zone_summary['eFG%'] = ((zone_summary['total_fgm'] + (0.5 * zone_summary['total_3pm'])) / zone_summary['total_fga']).round(3)
zone_summary['PPS'] = (zone_summary['total_points'] / zone_summary['total_fga']).round(3)

# Display final sorted summary table
zone_summary.sort_values(by='eFG%', ascending=False)

,spatial_zone,total_fga,total_fgm,total_3pm,total_points,eFG%,PPS
0,1. Restricted Area (0-4ft),289093,189362,0,378724,0.655,1.310
3,6. Above-the-Break 3,362539,130453,130453,391359,0.540,1.079
1,2. Paint / Short Mid (5-14ft),104068,49165,0,98330,0.472,0.945
2,3. Long Mid-Range (15ft+),186199,67770,0,135540,0.364,0.728


In [7]:
# Export your pre-aggregated spatial summary directly for Tableau
zone_summary.to_csv("spatial_zone_summary.csv", index=False)
print("Successfully exported spatial_zone_summary.csv for Tableau!")

Successfully exported spatial_zone_summary.csv for Tableau!
